# LoanLens - Data Cleaning

## Objective

The goal of this notebook is to clean the errors we found in the eda notebook where we made the analysis this will be a transformation notebook.

Cleaning will be performed in sequence so that we don't miss anything and preserve useful information for machine learning.

Dataset : application_train.csv

Expected Output : application_train_clean.csv


## Cleaning Plan
| Problem ID | Problem | Severity | Decision | Status |
|------------|----------|----------|----------|--------|
| P1 | Duplicate Rows | Low | Check & Remove | ⏳. |
| P2 | Missing Values | High | Analyze column-wise | ⏳ ..|
| P3 | Sentinel Values | High | Replace invalid values | ⏳ ...|
| P4 | Outliers | Medium | Decide treatment | ⏳ ....|
| P5 | Data Types | Low | Verify | ⏳..... |
| P6 | Constant Columns | Low | Remove if needed | ⏳...... |
| P7 | High Missing Columns | High | Decide drop/impute | ⏳....... |
| P8 | Redundant Features | Medium | Review | ⏳ ........|
| P9 | Target Leakage | High | Verify | ⏳......... |

# P1 first why are duplicate rows important 

### Why are duplicate rows important?

Duplicate records can bias statistical analysis and machine learning models by giving certain observations more importance than others.

During EDA, duplicate records were not explicitly removed, therefore they are verified before any further preprocessing.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", "{:.3f}".format)

df = pd.read_csv("../data/application_train.csv")

print(f"Dataset Shape : {df.shape}")

Dataset Shape : (307511, 122)


In [2]:
duplicate_rows = df.duplicated().sum()

print(f"Number of duplicate rows: {duplicate_rows}")

Number of duplicate rows: 0


### Observation

No duplicate records were found in the dataset.

### Cleaning Decision

No duplicate removal was required. The original dataset is retained without modification.
For a better understanding we performed it here as well it was already done in eda nootebook as well.

# P2 - Sentinel Values

## Problem

Some datasets use placeholder values to represent missing or unknown information instead of actual null values.

During the EDA phase, the `DAYS_EMPLOYED` feature was found to contain the value `365243`, which does not represent a realistic number of employment days.

Keeping this value would distort statistical analysis and negatively affect model training.

Therefore, these values will be replaced while preserving the information that they originally contained.

Let's see the verification below 

In [15]:
anomaly_count = (df['DAYS_EMPLOYED'] == 365243).sum()
print(f"Anomalous rows: {anomaly_count} ({anomaly_count/len(df)*100:.2f}%)")


Anomalous rows: 0 (0.00%)


In [ ]:
# create an anamoly flag is to preserve the information

df["DAYS_EMPLOYED_ANOM"] = ( df["DAYS_EMPLOYED"] == 365243 )

df["DAYS_EMPLOYED_ANOM"].value_counts()

In [12]:
df["DAYS_EMPLOYED"] = (df["DAYS_EMPLOYED"].replace(365243, np.nan)) #replacing the values with nan.

In [13]:
print("Remaining anomalous values:", (df["DAYS_EMPLOYED"] == 365243).sum())

df["DAYS_EMPLOYED"].describe()

Remaining anomalous values: 0


count   252137.000
mean     -2384.169
std       2338.360
min     -17912.000
25%      -3175.000
50%      -1648.000
75%       -767.000
max          0.000
Name: DAYS_EMPLOYED, dtype: float64

Over here when we used describe we found the employee day had exceptionally higher days which is not possible because 365243 days would mean that someone worked for 1000 years which is not possible. So, we changed the value with Nan values.

### Now we will go to the part where we will see does our missing vlaues means anything? why are they missing and how to deal with them? do they sing a different song than getting dropped?

#### Problem

Missing values are common in real-world datasets and can affect model performance if not handled appropriately.

However, not all missing values should be treated the same way. Some represent genuinely unavailable information, while others may carry meaningful information about an applicant.

Therefore, each feature group will be evaluated before selecting an appropriate cleaning strategy.

## Missing Value Strategy

| Missing Percentage | Strategy |
|--------------------|----------|
| 0% | No action required |
| Less than 5% | Impute |
| 5–30% | Review individually |
| More than 30% | Investigate before making a decision |

In [16]:
# Missing percentage for each column

missing_percent = (df.isnull().mean() * 100).sort_values(ascending=False)

missing_percent = missing_percent[missing_percent > 0]

missing_percent.head(20)

COMMONAREA_MEDI            69.872
COMMONAREA_AVG             69.872
COMMONAREA_MODE            69.872
NONLIVINGAPARTMENTS_MEDI   69.433
NONLIVINGAPARTMENTS_MODE   69.433
NONLIVINGAPARTMENTS_AVG    69.433
FONDKAPREMONT_MODE         68.386
LIVINGAPARTMENTS_AVG       68.355
LIVINGAPARTMENTS_MODE      68.355
LIVINGAPARTMENTS_MEDI      68.355
FLOORSMIN_MODE             67.849
FLOORSMIN_AVG              67.849
FLOORSMIN_MEDI             67.849
YEARS_BUILD_MEDI           66.498
YEARS_BUILD_AVG            66.498
YEARS_BUILD_MODE           66.498
OWN_CAR_AGE                65.991
LANDAREA_AVG               59.377
LANDAREA_MODE              59.377
LANDAREA_MEDI              59.377
dtype: float64

In [17]:
high_missing = missing_percent[missing_percent > 50]

medium_missing = missing_percent[
    (missing_percent > 5) &
    (missing_percent <= 50)
]

low_missing = missing_percent[missing_percent <= 5]

print(f"High Missing (>50%): {len(high_missing)} columns")
print(f"Medium Missing (5-50%): {len(medium_missing)} columns")
print(f"Low Missing (<5%): {len(low_missing)} columns")

High Missing (>50%): 41 columns
Medium Missing (5-50%): 17 columns
Low Missing (<5%): 10 columns


In [21]:
print(high_missing.index.tolist( ))

['COMMONAREA_MEDI', 'COMMONAREA_AVG', 'COMMONAREA_MODE', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAPARTMENTS_MODE', 'NONLIVINGAPARTMENTS_AVG', 'FONDKAPREMONT_MODE', 'LIVINGAPARTMENTS_AVG', 'LIVINGAPARTMENTS_MODE', 'LIVINGAPARTMENTS_MEDI', 'FLOORSMIN_MODE', 'FLOORSMIN_AVG', 'FLOORSMIN_MEDI', 'YEARS_BUILD_MEDI', 'YEARS_BUILD_AVG', 'YEARS_BUILD_MODE', 'OWN_CAR_AGE', 'LANDAREA_AVG', 'LANDAREA_MODE', 'LANDAREA_MEDI', 'BASEMENTAREA_MODE', 'BASEMENTAREA_MEDI', 'BASEMENTAREA_AVG', 'EXT_SOURCE_1', 'NONLIVINGAREA_AVG', 'NONLIVINGAREA_MODE', 'NONLIVINGAREA_MEDI', 'ELEVATORS_MODE', 'ELEVATORS_MEDI', 'ELEVATORS_AVG', 'WALLSMATERIAL_MODE', 'APARTMENTS_MODE', 'APARTMENTS_MEDI', 'APARTMENTS_AVG', 'ENTRANCES_AVG', 'ENTRANCES_MODE', 'ENTRANCES_MEDI', 'LIVINGAREA_MODE', 'LIVINGAREA_AVG', 'LIVINGAREA_MEDI', 'HOUSETYPE_MODE']


In [22]:
df[[
    "COMMONAREA_AVG",
    "COMMONAREA_MEDI",
    "COMMONAREA_MODE"
]].describe()

,COMMONAREA_AVG,COMMONAREA_MEDI,COMMONAREA_MODE
count,92646.000,92646.000,92646.000
mean,0.045,0.045,0.043
std,0.076,0.076,0.074
min,0.000,0.000,0.000
25%,0.008,0.008,0.007
50%,0.021,0.021,0.019
75%,0.051,0.051,0.049
max,1.000,1.000,1.000


In [23]:
df[[
    "COMMONAREA_AVG",
    "COMMONAREA_MEDI",
    "COMMONAREA_MODE"
]].head(10)

,COMMONAREA_AVG,COMMONAREA_MEDI,COMMONAREA_MODE
0,0.014,0.014,0.014
1,0.060,0.061,0.050
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


### Observation

The `COMMONAREA_AVG`, `COMMONAREA_MEDI`, and `COMMONAREA_MODE` features contain approximately **69.87% missing values**, with only **92,646** non-missing observations out of **307,511** records.

The descriptive statistics show that all three features have very similar distributions:

- Their means are nearly identical.
- Their quartiles (25%, 50%, and 75%) are almost the same.
- They all share the same minimum and maximum values.
- The missing values occur in exactly the same rows.

We can also see that over here the missigness is occuring together and when the values are pressent they are pretty close to each other. This shows these columns are representing the same underlying characteristic using different statistical summaries.

#### Decision 
At this stage, no cleaning operation will be applied to these features.

Although the columns contain a high percentage of missing values, they describe an important characteristic of the applicant's property. Since the three features represent different statistical summaries of the same attribute, they will be retained for now and evaluated later during feature engineering and feature selection.

The missing values will be handled during the preprocessing stage rather than being removed at this point.

In [24]:
df[[
    "NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAPARTMENTS_MODE"
]].describe()

,NONLIVINGAPARTMENTS_AVG,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAPARTMENTS_MODE
count,93997.000,93997.000,93997.000
mean,0.009,0.009,0.008
std,0.048,0.047,0.046
min,0.000,0.000,0.000
25%,0.000,0.000,0.000
50%,0.000,0.000,0.000
75%,0.004,0.004,0.004
max,1.000,1.000,1.000


In [25]:
df[[
    "NONLIVINGAPARTMENTS_AVG",
    "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAPARTMENTS_MODE"
]].head(10)

,NONLIVINGAPARTMENTS_AVG,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAPARTMENTS_MODE
0,0.000,0.000,0.000
1,0.004,0.004,0.000
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN
5,NaN,NaN,NaN
6,NaN,NaN,NaN
7,NaN,NaN,NaN
8,NaN,NaN,NaN
9,NaN,NaN,NaN


### Observation

The `NONLIVINGAPARTMENTS_AVG`, `NONLIVINGAPARTMENTS_MEDI`, and `NONLIVINGAPARTMENTS_MODE` features exhibit the same behaviour as the previously analysed `COMMONAREA` feature group.

The three columns:

- Have identical counts of non-missing observations.
- Share nearly identical descriptive statistics.
- Contain missing values in the same records.
- Represent the same underlying attribute using different statistical summaries (Average, Median, and Mode).

This indicates that the missingness is associated with the availability of the property information rather than an issue with individual columns.

#### Cleaning Decision

No cleaning operation will be applied at this stage.

The three features will be retained because they describe the same property using different statistical summaries. Since they contain potentially useful information, they will be evaluated later during feature selection and model development.

The missing values will be handled during the preprocessing stage.

In [26]:
pd.crosstab(
    df["FLAG_OWN_CAR"],
    df["OWN_CAR_AGE"].isna(),
    margins=True
)

OWN_CAR_AGE,False,True,All
FLAG_OWN_CAR,,,
N,0,202924,202924
Y,104582,5,104587
All,104582,202929,307511


### Observation

A cross-tabulation between **FLAG_OWN_CAR** and **OWN_CAR_AGE** shows that applicants who do not own a car (`FLAG_OWN_CAR = 'N'`) consistently have missing values for `OWN_CAR_AGE`.

This confirms that the missing values are expected and represent the absence of a car rather than missing or corrupted data.

#### Cleaning Decision

The `OWN_CAR_AGE` feature will be retained without replacing the missing values during the cleaning stage.

The missing values are meaningful because they indicate that an applicant does not own a car. Imputation will be considered later during preprocessing only if required by the selected machine learning algorithm.

In [29]:
df["EXT_SOURCE_1"].describe()

count   134133.000
mean         0.502
std          0.211
min          0.015
25%          0.334
50%          0.506
75%          0.675
max          0.963
Name: EXT_SOURCE_1, dtype: float64

In [30]:
df["EXT_SOURCE_1"].head(10)

0   0.083
1   0.311
2     NaN
3     NaN
4     NaN
5     NaN
6   0.775
7     NaN
8   0.587
9     NaN
Name: EXT_SOURCE_1, dtype: float64

In [31]:
print(f"Missing values: {df['EXT_SOURCE_1'].isnull().sum()}")
print(f"Missing percentage: {df['EXT_SOURCE_1'].isnull().mean()*100:.2f}%")

Missing values: 173378
Missing percentage: 56.38%


### Observation

The `EXT_SOURCE_1` feature contains **173,378** missing values (**56.38%** of the dataset).

Unlike the previously analysed property-related features, `EXT_SOURCE_1` is a single numerical feature with values ranging approximately between **0 and 1**, indicating that it represents a normalized external score.

The exact reason for the missing values cannot be determined from the dataset alone, therefore the missingness cannot be considered meaningful.

### Cleaning Decision

The `EXT_SOURCE_1` feature will be retained despite having a high percentage of missing values.

This feature is widely recognised as one of the most informative variables for predicting loan default in the Home Credit dataset. Removing it solely because of its missing percentage could significantly reduce model performance.

The missing values will be imputed later during the preprocessing stage using the machine learning pipeline.

In [32]:
df["OCCUPATION_TYPE"].describe()

count       211120
unique          18
top       Laborers
freq         55186
Name: OCCUPATION_TYPE, dtype: object

In [33]:
df["OCCUPATION_TYPE"].value_counts(dropna=False)

OCCUPATION_TYPE
NaN                      96391
Laborers                 55186
Sales staff              32102
Core staff               27570
Managers                 21371
Drivers                  18603
High skill tech staff    11380
Accountants               9813
Medicine staff            8537
Security staff            6721
Cooking staff             5946
Cleaning staff            4653
Private service staff     2652
Low-skill Laborers        2093
Waiters/barmen staff      1348
Secretaries               1305
Realty agents              751
HR staff                   563
IT staff                   526
Name: count, dtype: int64

### Observation

The `OCCUPATION_TYPE` feature contains **18 unique occupation categories** and **96,391 missing values** (approximately **31.35%** of the dataset).

The most frequent occupation is **Laborers**, while the missing values represent the largest single category.

Since occupation is an important socioeconomic characteristic, removing the feature or deleting rows with missing values could result in a significant loss of useful information.

#### Cleaning Decision

The `OCCUPATION_TYPE` feature will be retained.

The missing values will not be replaced during the cleaning stage. Instead, categorical imputation using the **most frequent category** will be performed later as part of the preprocessing pipeline before model training.

This approach avoids data leakage and ensures a consistent preprocessing workflow.

In [34]:
low_missing

NAME_TYPE_SUITE            0.420
DEF_60_CNT_SOCIAL_CIRCLE   0.332
OBS_60_CNT_SOCIAL_CIRCLE   0.332
OBS_30_CNT_SOCIAL_CIRCLE   0.332
DEF_30_CNT_SOCIAL_CIRCLE   0.332
EXT_SOURCE_2               0.215
AMT_GOODS_PRICE            0.090
AMT_ANNUITY                0.004
CNT_FAM_MEMBERS            0.001
DAYS_LAST_PHONE_CHANGE     0.000
dtype: float64

### Cleaning Decision

These features will be retained without any modifications during the data cleaning stage.

Since the proportion of missing values is negligible, the missing numerical values will be imputed using the **median**, while the missing categorical values will be imputed using the **most frequent category** during the preprocessing stage.

Performing imputation within the preprocessing pipeline prevents data leakage and ensures that the same transformations are consistently applied to both the training and testing datasets.

#### Cleaning Decision

These features will be retained without any modifications during the data cleaning stage.

Since the proportion of missing values is negligible, the missing numerical values will be imputed using the **median**, while the missing categorical values will be imputed using the **most frequent category** during the preprocessing stage.

Performing imputation within the preprocessing pipeline prevents data leakage and ensures that the same transformations are consistently applied to both the training and testing datasets.   